# Parkinson's Drawing Model Comparison

A participant-level comparison of classical and deep-learning image classifiers using the NewHandPD drawing dataset.

## Setup

In [ ]:
import hashlib
import platform
from pathlib import Path
from time import perf_counter
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps
from skimage.feature import hog
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd()

print(f"Python: {platform.python_version()}")
print(f"Project root: {PROJECT_ROOT}")

## Download

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
NEWHANDPD_DIR = DATA_DIR / "raw" / "NewHandPD"
ARCHIVE_DIR = NEWHANDPD_DIR / "archives"
DOWNLOAD_MARKER = NEWHANDPD_DIR / ".download_complete"

ARCHIVES = {
    ("Healthy", "circle"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthyCircle.zip",
    ("Healthy", "meander"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthyMeander.zip",
    ("Healthy", "spiral"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthySpiral.zip",
    ("PD", "circle"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientCircle.zip",
    ("PD", "meander"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientMeander.zip",
    ("PD", "spiral"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientSpiral.zip",
}

download_required = not DOWNLOAD_MARKER.exists()

if download_required:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

    for (label, drawing_type), url in ARCHIVES.items():
        archive_path = ARCHIVE_DIR / f"{label.lower()}_{drawing_type}.zip"
        extraction_dir = NEWHANDPD_DIR / label / drawing_type
        extraction_dir.mkdir(parents=True, exist_ok=True)

        if not archive_path.exists():
            temporary_path = archive_path.with_suffix(".zip.part")
            print(f"Downloading {archive_path.name}...")
            urlretrieve(url, temporary_path)
            temporary_path.replace(archive_path)

        with ZipFile(archive_path) as archive:
            members = [
                member
                for member in archive.infolist()
                if "__MACOSX" not in Path(member.filename).parts
                and not Path(member.filename).name.startswith("._")
            ]
            extraction_root = extraction_dir.resolve()
            member_paths = [
                (extraction_dir / member.filename).resolve()
                for member in members
            ]
            if any(not path.is_relative_to(extraction_root) for path in member_paths):
                raise ValueError(f"Unsafe path found in {archive_path.name}")
            archive.extractall(extraction_dir, members=members)

image_suffixes = {".jpg", ".jpeg", ".png"}
image_paths = sorted(
    path
    for path in NEWHANDPD_DIR.rglob("*")
    if path.suffix.lower() in image_suffixes
    and "__MACOSX" not in path.parts
    and not path.name.startswith("._")
)

if len(image_paths) != 594:
    raise RuntimeError(f"Expected 594 NewHandPD images, found {len(image_paths):,}")

if download_required:
    DOWNLOAD_MARKER.touch()

print(f"Dataset folder: {NEWHANDPD_DIR}")
print(f"Downloaded now: {download_required}")
print(f"Images found: {len(image_paths):,}")
image_paths[:5]

## Inspect data

In [ ]:
if not NEWHANDPD_DIR.is_dir():
    raise FileNotFoundError(f"NewHandPD folder not found: {NEWHANDPD_DIR}")

records = []
invalid_image_paths = []

for image_path in image_paths:
    name_parts = image_path.stem.rsplit("-", maxsplit=1)
    if len(name_parts) != 2:
        invalid_image_paths.append(image_path)
        continue

    task, source_participant_code = name_parts
    participant_number = source_participant_code[1:]

    label = image_path.relative_to(NEWHANDPD_DIR).parts[0]

    if (
        label not in {"Healthy", "PD"}
        or not task[:1].isalpha()
        or not participant_number.isdigit()
    ):
        invalid_image_paths.append(image_path)
        continue

    records.append(
        {
            "path": image_path,
            "label": label,
            "participant_id": f"{label.lower()}-{int(participant_number):02d}",
            "source_participant_code": source_participant_code.upper(),
            "task": task.lower(),
        }
    )

manifest = (
    pd.DataFrame.from_records(records)
    .sort_values(["label", "participant_id", "task"])
    .reset_index(drop=True)
)

class_summary = (
    manifest.groupby("label")
    .agg(images=("path", "size"), participants=("participant_id", "nunique"))
    .sort_index()
)
task_summary = (
    manifest.groupby(["task", "label"])
    .size()
    .unstack(fill_value=0)
)

print(f"NewHandPD images: {len(manifest):,}")
print(f"Invalid image filenames: {len(invalid_image_paths):,}")
display(class_summary)
display(task_summary)
manifest.head()

## Validate images

In [ ]:
image_metadata_records = []
invalid_image_records = []

for row in manifest.itertuples(index=False):
    try:
        with Image.open(row.path) as image:
            image_format = image.format
            image_mode = image.mode
            width, height = image.size
            image.verify()

        with row.path.open("rb") as image_file:
            sha256 = hashlib.file_digest(image_file, "sha256").hexdigest()

        image_metadata_records.append(
            {
                "path": row.path,
                "format": image_format,
                "mode": image_mode,
                "width": width,
                "height": height,
                "sha256": sha256,
            }
        )
    except (OSError, SyntaxError) as error:
        invalid_image_records.append(
            {"path": row.path, "error": str(error)}
        )

invalid_images = pd.DataFrame(
    invalid_image_records,
    columns=["path", "error"],
)

if not invalid_images.empty:
    display(invalid_images)
    raise RuntimeError(f"Image validation failed for {len(invalid_images)} files")

image_metadata = pd.DataFrame.from_records(image_metadata_records)
validated_manifest = manifest.merge(
    image_metadata,
    on="path",
    how="left",
    validate="one_to_one",
)

image_profiles = (
    validated_manifest.groupby(["format", "mode", "width", "height"])
    .size()
    .rename("images")
    .reset_index()
    .sort_values("images", ascending=False)
)

duplicate_images = validated_manifest[
    validated_manifest.duplicated("sha256", keep=False)
].sort_values(["sha256", "label", "participant_id", "task"])
duplicate_summary = (
    duplicate_images.groupby("sha256")
    .agg(
        files=("path", "size"),
        participants=("participant_id", "nunique"),
        labels=("label", "nunique"),
        tasks=("task", "nunique"),
    )
    .reset_index()
)

expected_tasks = {"circa", "mea1", "mea2", "mea3", "mea4", "sp1", "sp2", "sp3", "sp4"}
participant_task_sets = validated_manifest.groupby("participant_id")["task"].agg(set)
task_issue_records = []

for participant_id, observed_tasks in participant_task_sets.items():
    missing_tasks = sorted(expected_tasks - observed_tasks)
    unexpected_tasks = sorted(observed_tasks - expected_tasks)
    if missing_tasks or unexpected_tasks:
        task_issue_records.append(
            {
                "participant_id": participant_id,
                "missing_tasks": missing_tasks,
                "unexpected_tasks": unexpected_tasks,
            }
        )

task_issues = pd.DataFrame(
    task_issue_records,
    columns=["participant_id", "missing_tasks", "unexpected_tasks"],
)

expected_prefix = {"Healthy": "H", "PD": "P"}
participant_code_issues = validated_manifest[
    validated_manifest.apply(
        lambda row: row.source_participant_code[:1] != expected_prefix[row.label],
        axis=1,
    )
]["path label participant_id source_participant_code task".split()]

print(f"Images validated: {len(validated_manifest):,}")
print(f"Unreadable images: {len(invalid_images):,}")
print(f"Exact duplicate groups: {len(duplicate_summary):,}")
print(f"Duplicate groups spanning participants: {(duplicate_summary['participants'] > 1).sum():,}")
print(f"Duplicate groups spanning labels: {(duplicate_summary['labels'] > 1).sum():,}")
print(f"Participants with task issues: {len(task_issues):,}")
print(f"Participant-code prefix issues: {len(participant_code_issues):,}")
display(image_profiles.head(10))
display(task_issues)
participant_code_issues.head()

## Clean manifest

The source file `mea5-P8.jpg` is retained unchanged and its filename-derived task remains available as `source_task`. Because participant P8 has `mea1`, `mea2`, `mea3`, and `mea5` but no `mea4`, while the dataset structure contains four meander repetitions, the analytical `task` value is normalised from `mea5` to `mea4`. This is an explicit inference from the dataset structure, not an author-confirmed correction.

In [ ]:
clean_manifest = validated_manifest.copy()
clean_manifest = clean_manifest.rename(columns={"task": "source_task"})
clean_manifest["task"] = clean_manifest["source_task"]

task_normalisations = {
    ("pd-08", "mea5"): "mea4",
}

for (participant_id, source_task), normalised_task in task_normalisations.items():
    normalisation_mask = (
        clean_manifest["participant_id"].eq(participant_id)
        & clean_manifest["source_task"].eq(source_task)
    )
    if normalisation_mask.sum() != 1:
        raise RuntimeError(
            f"Expected one {source_task} image for {participant_id}, "
            f"found {normalisation_mask.sum()}"
        )
    clean_manifest.loc[normalisation_mask, "task"] = normalised_task

participant_numbers = (
    clean_manifest["participant_id"].str.rsplit("-", n=1).str[-1].astype(int)
)
participant_prefixes = clean_manifest["label"].map({"Healthy": "H", "PD": "P"})
clean_manifest["participant_code"] = participant_prefixes + participant_numbers.astype(str)
clean_manifest["participant_code_normalised"] = (
    clean_manifest["participant_code"] != clean_manifest["source_participant_code"]
)
clean_manifest["task_normalised"] = (
    clean_manifest["task"] != clean_manifest["source_task"]
)
clean_manifest["is_exact_duplicate"] = clean_manifest.duplicated(
    "sha256",
    keep=False,
)

parent = {
    participant_id: participant_id
    for participant_id in clean_manifest["participant_id"].unique()
}

def find_group(participant_id):
    while parent[participant_id] != participant_id:
        parent[participant_id] = parent[parent[participant_id]]
        participant_id = parent[participant_id]
    return participant_id

def join_groups(first_participant, second_participant):
    first_root = find_group(first_participant)
    second_root = find_group(second_participant)
    if first_root != second_root:
        parent[second_root] = first_root

duplicate_rows = clean_manifest[clean_manifest["is_exact_duplicate"]]
for _, duplicate_group in duplicate_rows.groupby("sha256"):
    participants = sorted(duplicate_group["participant_id"].unique())
    for participant_id in participants[1:]:
        join_groups(participants[0], participant_id)

clean_manifest["split_group"] = clean_manifest["participant_id"].map(find_group)

normalised_task_sets = clean_manifest.groupby("participant_id")["task"].agg(set)
if not normalised_task_sets.map(lambda tasks: tasks == expected_tasks).all():
    raise RuntimeError("Task normalisation did not produce nine expected tasks per participant")

duplicate_split_counts = duplicate_rows.assign(
    split_group=duplicate_rows["participant_id"].map(find_group)
).groupby("sha256")["split_group"].nunique()
if not duplicate_split_counts.eq(1).all():
    raise RuntimeError("An exact duplicate group spans multiple split groups")

print(f"Task labels normalised under the documented rule: {clean_manifest['task_normalised'].sum():,}")
print(f"Participant codes normalised from class and participant number: {clean_manifest['participant_code_normalised'].sum():,}")
print(f"Images in exact duplicate groups: {clean_manifest['is_exact_duplicate'].sum():,}")
print(f"Participants: {clean_manifest['participant_id'].nunique():,}")
print(f"Leakage-safe split groups: {clean_manifest['split_group'].nunique():,}")
clean_manifest[
    [
        "path",
        "label",
        "participant_id",
        "participant_code",
        "task",
        "split_group",
        "is_exact_duplicate",
    ]
].head()

## Preview images

Display all nine drawings for participant 8 in each class. The preview uses the analytical task labels while preserving any different filename-derived label in the caption.

In [ ]:
preview_participants = {
    "Healthy": "healthy-08",
    "PD": "pd-08",
}
preview_tasks = ["circa", "mea1", "mea2", "mea3", "mea4", "sp1", "sp2", "sp3", "sp4"]
thumbnail_size = (180, 180)
caption_height = 24
columns = 3
rows = 3
gap = 10

for label, participant_id in preview_participants.items():
    participant_images = (
        clean_manifest.loc[
            clean_manifest["label"].eq(label)
            & clean_manifest["participant_id"].eq(participant_id)
        ]
        .set_index("task")
        .loc[preview_tasks]
    )

    canvas_width = columns * thumbnail_size[0] + (columns + 1) * gap
    canvas_height = rows * (caption_height + thumbnail_size[1]) + (rows + 1) * gap
    canvas = Image.new("RGB", (canvas_width, canvas_height), "white")
    draw = ImageDraw.Draw(canvas)

    for index, (task, image_row) in enumerate(participant_images.iterrows()):
        with Image.open(image_row["path"]) as source_image:
            thumbnail = ImageOps.contain(source_image.convert("RGB"), thumbnail_size)

        column = index % columns
        grid_row = index // columns
        x = gap + column * (thumbnail_size[0] + gap)
        y = gap + grid_row * (caption_height + thumbnail_size[1] + gap)
        image_x = x + (thumbnail_size[0] - thumbnail.width) // 2
        image_y = y + caption_height + (thumbnail_size[1] - thumbnail.height) // 2
        canvas.paste(thumbnail, (image_x, image_y))

        source_task = image_row["source_task"]
        caption = task if source_task == task else f"{task} (source: {source_task})"
        draw.text((x, y), caption, fill="black")

    participant_code = participant_images.iloc[0]["participant_code"]
    print(f"{label}: {participant_id} ({participant_code})")
    display(canvas)

## Split data

Reserve one fold as a fixed test set before preprocessing or modelling. Five-fold stratified group splitting gives an approximately 80/20 division while keeping each `split_group` wholly within one partition. The exact proportion can differ because duplicate-linked participant groups are indivisible.

In [ ]:
random_state = 42
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)

train_indices, test_indices = next(
    splitter.split(
        clean_manifest,
        y=clean_manifest["label"],
        groups=clean_manifest["split_group"],
    )
)

split_manifest = clean_manifest.copy()
split_manifest["partition"] = "train"
split_manifest.loc[split_manifest.index[test_indices], "partition"] = "test"

participant_partition_counts = (
    split_manifest.groupby("participant_id")["partition"].nunique()
)
if not participant_partition_counts.eq(1).all():
    raise RuntimeError("A participant appears in both train and test partitions")

group_partition_counts = split_manifest.groupby("split_group")["partition"].nunique()
if not group_partition_counts.eq(1).all():
    raise RuntimeError("A leakage-safe split group appears in both partitions")

duplicate_partition_counts = (
    split_manifest.loc[split_manifest["is_exact_duplicate"]]
    .groupby("sha256")["partition"]
    .nunique()
)
if not duplicate_partition_counts.eq(1).all():
    raise RuntimeError("Exact duplicate content appears in both partitions")

labels_per_partition = split_manifest.groupby("partition")["label"].nunique()
if not labels_per_partition.eq(clean_manifest["label"].nunique()).all():
    raise RuntimeError("A partition does not contain both diagnostic classes")

split_summary = (
    split_manifest.groupby(["partition", "label"])
    .agg(
        images=("path", "size"),
        participants=("participant_id", "nunique"),
        split_groups=("split_group", "nunique"),
    )
    .sort_index()
)

print(f"Random state: {random_state}")
print("Participant overlap: 0")
print("Split-group overlap: 0")
print("Exact-duplicate overlap: 0")
display(split_summary)

## Preprocess images

Apply the same minimal preprocessing to every image: correct any stored orientation, convert to greyscale, resize to fit within `128 × 128` pixels without changing the aspect ratio, centre on a white square, and scale pixel intensities to `[0, 1]`. Processing is performed in memory; the source files remain unchanged and no resized copies are written to disk.

In [ ]:
image_size = (128, 128)

def preprocess_image(image_path, output_size=image_size):
    with Image.open(image_path) as source_image:
        greyscale_image = ImageOps.exif_transpose(source_image).convert("L")
        resized_image = ImageOps.contain(
            greyscale_image,
            output_size,
            method=Image.Resampling.LANCZOS,
        )

    processed_image = Image.new("L", output_size, color=255)
    offset = (
        (output_size[0] - resized_image.width) // 2,
        (output_size[1] - resized_image.height) // 2,
    )
    processed_image.paste(resized_image, offset)
    return np.asarray(processed_image, dtype=np.float32) / 255.0

preprocessed_images = np.stack(
    [preprocess_image(image_path) for image_path in split_manifest["path"]],
)
split_manifest = split_manifest.copy()
split_manifest["image_index"] = np.arange(len(split_manifest))

expected_shape = (len(split_manifest), image_size[1], image_size[0])
if preprocessed_images.shape != expected_shape:
    raise RuntimeError(
        f"Expected preprocessed shape {expected_shape}, found {preprocessed_images.shape}"
    )
if not np.isfinite(preprocessed_images).all():
    raise RuntimeError("Preprocessed images contain non-finite values")
if preprocessed_images.min() < 0.0 or preprocessed_images.max() > 1.0:
    raise RuntimeError("Preprocessed pixel values fall outside [0, 1]")

preprocessing_summary = pd.Series(
    {
        "images": len(preprocessed_images),
        "array shape": str(preprocessed_images.shape),
        "data type": str(preprocessed_images.dtype),
        "minimum pixel value": float(preprocessed_images.min()),
        "maximum pixel value": float(preprocessed_images.max()),
        "memory (MiB)": preprocessed_images.nbytes / (1024 ** 2),
    },
    name="value",
)
display(preprocessing_summary)

## Extract HOG features

A Histogram of Oriented Gradients (HOG) does not classify an image. It converts the image into measurements that a classical classifier can use. The image is divided into small cells; each cell records the directions of its strongest brightness changes, which correspond to local edges and stroke directions. Neighbouring cells are then normalised together to reduce sensitivity to contrast.

Here, each `128 × 128` image is divided into `8 × 8`-pixel cells, edge directions are represented by nine bins, and neighbouring `2 × 2` cells are normalised together. The fixed, label-free transformation is applied identically to the training and test images.

In [ ]:
hog_settings = {
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2),
    "block_norm": "L2-Hys",
    "transform_sqrt": False,
    "feature_vector": True,
    "channel_axis": None,
}

def extract_hog_features(image):
    return hog(image, **hog_settings).astype(np.float32)

hog_feature_matrix = np.stack(
    [extract_hog_features(image) for image in preprocessed_images],
)

if hog_feature_matrix.shape[0] != len(split_manifest):
    raise RuntimeError("HOG feature rows do not match the manifest rows")
if not np.isfinite(hog_feature_matrix).all():
    raise RuntimeError("HOG features contain non-finite values")

example_index = int(
    split_manifest.loc[
        split_manifest["participant_id"].eq("healthy-08")
        & split_manifest["task"].eq("sp1"),
        "image_index",
    ].iloc[0]
)
example_features, hog_visualisation = hog(
    preprocessed_images[example_index],
    visualize=True,
    **hog_settings,
)
if not np.allclose(example_features, hog_feature_matrix[example_index]):
    raise RuntimeError("HOG visualisation call produced different feature values")

visualisation_range = np.ptp(hog_visualisation)
if visualisation_range == 0:
    raise RuntimeError("HOG visualisation has no intensity variation")
scaled_visualisation = (
    (hog_visualisation - hog_visualisation.min()) / visualisation_range * 255
).astype(np.uint8)

hog_preview = Image.new("L", (256, 152), color=255)
hog_preview.paste(
    Image.fromarray((preprocessed_images[example_index] * 255).astype(np.uint8)),
    (0, 24),
)
hog_preview.paste(Image.fromarray(scaled_visualisation), (128, 24))
preview_draw = ImageDraw.Draw(hog_preview)
preview_draw.text((4, 6), "Preprocessed image", fill=0)
preview_draw.text((132, 6), "HOG visualisation", fill=0)

hog_summary = pd.Series(
    {
        "images": hog_feature_matrix.shape[0],
        "features per image": hog_feature_matrix.shape[1],
        "matrix shape": str(hog_feature_matrix.shape),
        "data type": str(hog_feature_matrix.dtype),
        "memory (MiB)": hog_feature_matrix.nbytes / (1024 ** 2),
    },
    name="value",
)
display(hog_preview)
display(hog_summary)

## Train the HOG–SVM model

An SVM learns a boundary between the Healthy and Parkinson's classes using the HOG measurements rather than the original pixels. The parameter `C` controls how strongly the model penalises training errors. A linear kernel learns one straight boundary in the HOG feature space; an RBF kernel can learn a curved boundary.

Six candidate settings are compared with five-fold stratified group cross-validation inside the training partition. Every validation fold contains participants and duplicate-linked groups that are absent from its corresponding fitting fold. Scores from the nine drawings are averaged for each participant, and balanced accuracy gives equal importance to the two diagnostic classes. The fixed test partition is not used during this process.

In [ ]:
label_to_target = {"Healthy": 0, "PD": 1}
training_mask = split_manifest["partition"].eq("train").to_numpy()
X_hog_train = hog_feature_matrix[training_mask]
training_metadata = split_manifest.loc[training_mask].reset_index(drop=True)
y_train = training_metadata["label"].map(label_to_target).to_numpy()
training_groups = training_metadata["split_group"].to_numpy()

cv_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)
cv_splits = list(cv_splitter.split(X_hog_train, y_train, groups=training_groups))

for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
    cv_splits,
    start=1,
):
    fold_train_groups = set(training_groups[fold_train_indices])
    fold_validation_groups = set(training_groups[fold_validation_indices])
    if fold_train_groups & fold_validation_groups:
        raise RuntimeError(f"Split-group leakage detected in cross-validation fold {fold_number}")

svm_candidates = [
    {"kernel": "linear", "C": 0.1},
    {"kernel": "linear", "C": 1.0},
    {"kernel": "linear", "C": 10.0},
    {"kernel": "rbf", "C": 0.1, "gamma": "scale"},
    {"kernel": "rbf", "C": 1.0, "gamma": "scale"},
    {"kernel": "rbf", "C": 10.0, "gamma": "scale"},
]

cv_fold_results = []

for candidate_id, candidate_parameters in enumerate(svm_candidates):
    for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
        cv_splits,
        start=1,
    ):
        fold_model = Pipeline(
            [
                ("standardise", StandardScaler()),
                (
                    "svm",
                    SVC(
                        class_weight="balanced",
                        **candidate_parameters,
                    ),
                ),
            ]
        )
        fold_model.fit(X_hog_train[fold_train_indices], y_train[fold_train_indices])

        validation_scores = fold_model.decision_function(
            X_hog_train[fold_validation_indices]
        )
        validation_image_predictions = (validation_scores >= 0).astype(int)
        validation_metadata = training_metadata.iloc[fold_validation_indices].copy()
        validation_metadata["decision_score"] = validation_scores

        participant_validation = (
            validation_metadata.groupby(["participant_id", "label"], as_index=False)
            .agg(decision_score=("decision_score", "mean"))
        )
        participant_targets = (
            participant_validation["label"].map(label_to_target).to_numpy()
        )
        participant_predictions = (
            participant_validation["decision_score"].to_numpy() >= 0
        ).astype(int)

        cv_fold_results.append(
            {
                "candidate_id": candidate_id,
                "kernel": candidate_parameters["kernel"],
                "C": candidate_parameters["C"],
                "gamma": candidate_parameters.get("gamma", "not used"),
                "fold": fold_number,
                "image_balanced_accuracy": balanced_accuracy_score(
                    y_train[fold_validation_indices],
                    validation_image_predictions,
                ),
                "participant_balanced_accuracy": balanced_accuracy_score(
                    participant_targets,
                    participant_predictions,
                ),
            }
        )

cv_fold_results = pd.DataFrame(cv_fold_results)
cv_summary = (
    cv_fold_results.groupby(
        ["candidate_id", "kernel", "C", "gamma"],
        as_index=False,
    )
    .agg(
        participant_balanced_accuracy_mean=(
            "participant_balanced_accuracy",
            "mean",
        ),
        participant_balanced_accuracy_std=(
            "participant_balanced_accuracy",
            "std",
        ),
        image_balanced_accuracy_mean=("image_balanced_accuracy", "mean"),
    )
    .sort_values(
        [
            "participant_balanced_accuracy_mean",
            "participant_balanced_accuracy_std",
            "candidate_id",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

best_candidate_id = int(cv_summary.loc[0, "candidate_id"])
best_svm_parameters = svm_candidates[best_candidate_id]
best_hog_svm = Pipeline(
    [
        ("standardise", StandardScaler()),
        (
            "svm",
            SVC(
                class_weight="balanced",
                **best_svm_parameters,
            ),
        ),
    ]
)
best_hog_svm.fit(X_hog_train, y_train)

print(f"Training images used: {len(X_hog_train):,}")
print(f"Training participants used: {training_metadata['participant_id'].nunique():,}")
print("Test images used for fitting or tuning: 0")
print(f"Best SVM parameters: {best_svm_parameters}")
display(cv_summary.round(3))

## Evaluate the HOG–SVM model

Evaluate the fixed model on the held-out test partition without further fitting or parameter selection. Drawing-level metrics treat every image as a separate prediction. The primary participant-level metrics average the nine drawing decision scores for each person before assigning a diagnosis. Parkinson's is the positive class. Balanced accuracy is the mean of Parkinson's sensitivity and Healthy specificity, so the two classes contribute equally.

In [ ]:
def calculate_binary_metrics(targets, predictions, decision_scores):
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        targets,
        predictions,
        labels=[0, 1],
    ).ravel()
    return {
        "accuracy": accuracy_score(targets, predictions),
        "balanced_accuracy": balanced_accuracy_score(targets, predictions),
        "sensitivity_pd": recall_score(targets, predictions, pos_label=1),
        "specificity_healthy": true_negative / (true_negative + false_positive),
        "precision_pd": precision_score(
            targets,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "f1_pd": f1_score(
            targets,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(targets, decision_scores),
    }

test_mask = split_manifest["partition"].eq("test").to_numpy()
X_hog_test = hog_feature_matrix[test_mask]
test_metadata = split_manifest.loc[test_mask].reset_index(drop=True)
y_test = test_metadata["label"].map(label_to_target).to_numpy()

test_decision_scores = best_hog_svm.decision_function(X_hog_test)
test_image_predictions = (test_decision_scores >= 0).astype(int)
test_image_results = test_metadata[
    ["participant_id", "label", "task", "split_group"]
].copy()
test_image_results["target"] = y_test
test_image_results["decision_score"] = test_decision_scores
test_image_results["prediction"] = test_image_predictions

test_participant_results = (
    test_image_results.groupby(["participant_id", "label"], as_index=False)
    .agg(
        images=("task", "size"),
        decision_score=("decision_score", "mean"),
    )
)
test_participant_results["target"] = (
    test_participant_results["label"].map(label_to_target).astype(int)
)
test_participant_results["prediction"] = (
    test_participant_results["decision_score"] >= 0
).astype(int)
test_participant_results["predicted_label"] = (
    test_participant_results["prediction"].map({0: "Healthy", 1: "PD"})
)
test_participant_results["correct"] = (
    test_participant_results["target"]
    == test_participant_results["prediction"]
)

evaluation_metrics = pd.DataFrame(
    {
        "drawing level": calculate_binary_metrics(
            y_test,
            test_image_predictions,
            test_decision_scores,
        ),
        "participant level": calculate_binary_metrics(
            test_participant_results["target"],
            test_participant_results["prediction"],
            test_participant_results["decision_score"],
        ),
    }
).T

participant_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        test_participant_results["target"],
        test_participant_results["prediction"],
        labels=[0, 1],
    ),
    index=["Actual Healthy", "Actual PD"],
    columns=["Predicted Healthy", "Predicted PD"],
)

if not test_participant_results["images"].eq(len(expected_tasks)).all():
    raise RuntimeError("A test participant does not have nine evaluated drawings")
if len(test_participant_results) != test_metadata["participant_id"].nunique():
    raise RuntimeError("Participant-level aggregation changed the participant count")

print(f"Held-out test images: {len(test_metadata):,}")
print(f"Held-out test participants: {len(test_participant_results):,}")
display(evaluation_metrics.round(3))
display(participant_confusion_matrix)
display(
    test_participant_results[
        [
            "participant_id",
            "label",
            "decision_score",
            "predicted_label",
            "correct",
        ]
    ].sort_values("decision_score")
)

## Construct the CNN

The CNN receives the same preprocessed greyscale images as the classical model. During training, its convolution filters will learn local patterns directly from pixels. PyTorch makes these layers and their learned parameters explicit.

Three stages use 16, 32 and 64 filters with `3 × 3` kernels. Each stage applies ReLU, which sets negative responses to zero, and max pooling, which reduces spatial size by keeping the strongest response in each `2 × 2` region. Adaptive average pooling then reduces each feature map to a `4 × 4` grid, retaining a coarse spatial layout while limiting the size of the final classifier. A 32-unit layer combines those measurements; dropout randomly masks 25% of its activations during training to discourage reliance on individual units.

The output is one raw score (a logit) per drawing, with higher values corresponding to the PD target of 1 once trained. It is not a calibrated probability. This is a starting architecture with randomly initialised weights. The code below checks its dimensions using four training images; no learning occurs in this section.

In [ ]:
def build_cnn(seed):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
        nn.AdaptiveAvgPool2d((4, 4)),
        nn.Flatten(),
        nn.Linear(64 * 4 * 4, 32),
        nn.ReLU(),
        nn.Dropout(p=0.25),
        nn.Linear(32, 1),
    )


cnn_model = build_cnn(seed=random_state)
cnn_example_indices = (
    split_manifest.loc[split_manifest["partition"].eq("train"), "image_index"]
    .head(4)
    .to_numpy()
)
cnn_example_images = torch.from_numpy(
    preprocessed_images[cnn_example_indices]
).unsqueeze(1)

# PyTorch expects (batch, channels, height, width); greyscale has one channel.
expected_cnn_input_shape = (
    len(cnn_example_indices), 1, image_size[1], image_size[0]
)
if not len(cnn_example_indices):
    raise RuntimeError("No training images are available for the CNN shape check")
if tuple(cnn_example_images.shape) != expected_cnn_input_shape:
    raise RuntimeError("CNN input dimensions do not match the preprocessed images")

cnn_layer_rows = []
cnn_model.eval()
with torch.no_grad():
    cnn_example_logits = cnn_model(cnn_example_images)
    layer_output = cnn_example_images
    for layer_number, layer in enumerate(cnn_model, start=1):
        layer_output = layer(layer_output)
        cnn_layer_rows.append(
            {
                "layer": layer_number,
                "operation": str(layer),
                "output shape": tuple(layer_output.shape),
                "trainable parameters": sum(
                    parameter.numel()
                    for parameter in layer.parameters()
                    if parameter.requires_grad
                ),
            }
        )
cnn_model.train()

if tuple(cnn_example_logits.shape) != (len(cnn_example_indices), 1):
    raise RuntimeError("The CNN must produce one logit per drawing")
if not torch.isfinite(cnn_example_logits).all():
    raise RuntimeError("The CNN produced non-finite logits")

cnn_layer_summary = pd.DataFrame(cnn_layer_rows)
cnn_parameter_count = sum(
    parameter.numel()
    for parameter in cnn_model.parameters()
    if parameter.requires_grad
)

print(f"PyTorch: {torch.__version__}")
print(f"Initialisation seed: {random_state}")
print(f"Input shape (batch, channels, height, width): {tuple(cnn_example_images.shape)}")
print(f"Output shape (batch, logit): {tuple(cnn_example_logits.shape)}")
print(f"Trainable parameters: {cnn_parameter_count:,}")
print("Construction check passed; the CNN has not been trained.")
display(cnn_layer_summary)

## Train the CNN

Training adjusts the CNN's weights to reduce prediction error. Adam updates those weights in batches of 32 drawings; its learning rate controls the size of each update. An epoch is one pass through the fitting images. Binary cross-entropy measures the error from raw logits, with the PD weight calculated as Healthy drawings divided by PD drawings in the current fitting subset. Weight decay of `0.0001` penalises large weights.

The search compares learning rates `0.0003` and `0.001` at 10, 20 and 30 epochs, using exactly the SVM's five training folds. Each learning-rate/fold run starts with fresh weights and records the three scheduled checkpoints. Seeds are paired across learning rates within each fold, and each participant and duplicate-linked group stays in one fold partition. The image tensors follow the manifest's explicit `image_index` mapping.

Selection uses mean participant-level balanced accuracy after averaging each person's nine logits and applying the fixed zero threshold. Ties favour lower fold-to-fold standard deviation, then candidate order. The selected learning rate and common epoch count are used to train a fresh model on all training participants. These cross-validation scores are used for selection and are not final test estimates. Training runs on the CPU with fixed seeds and deterministic operations; exact reproducibility is limited to a compatible software and hardware environment.

In [ ]:
cnn_learning_rates = [0.0003, 0.001]
cnn_epoch_options = [10, 20, 30]
cnn_batch_size = 32
cnn_weight_decay = 0.0001
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.use_deterministic_algorithms(True)

cnn_training_images = torch.from_numpy(
    preprocessed_images[training_metadata["image_index"].to_numpy()]
).unsqueeze(1)
cnn_training_targets = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

if not training_metadata["partition"].eq("train").all():
    raise RuntimeError("CNN fitting data includes a held-out test image")
if not np.array_equal(
    training_metadata["label"].map(label_to_target).to_numpy(), y_train
):
    raise RuntimeError("CNN targets do not align with the training metadata")

cnn_fold_rows = []
cnn_validation_visits = np.zeros(len(training_metadata), dtype=int)
for fold_number, (fit_indices, validation_indices) in enumerate(cv_splits, start=1):
    fit_metadata = training_metadata.iloc[fit_indices]
    validation_metadata = training_metadata.iloc[validation_indices]
    for column in ["participant_id", "split_group", "sha256"]:
        if set(fit_metadata[column]) & set(validation_metadata[column]):
            raise RuntimeError(f"{column} overlap in CNN fold {fold_number}")
    if fit_metadata["label"].nunique() != 2 or validation_metadata["label"].nunique() != 2:
        raise RuntimeError(f"Both classes are required in CNN fold {fold_number}")
    cnn_validation_visits[validation_indices] += 1
    cnn_fold_rows.append(
        {
            "fold": fold_number,
            "fitting participants": fit_metadata["participant_id"].nunique(),
            "validation participants": validation_metadata["participant_id"].nunique(),
            "fitting images": len(fit_metadata),
            "validation images": len(validation_metadata),
        }
    )
if not (cnn_validation_visits == 1).all():
    raise RuntimeError("Each training image must appear in one CNN validation fold")


def fit_cnn(images, targets, *, learning_rate, epochs, seed,
            validation_data=None, progress_label="CNN"):
    model = build_cnn(seed=seed)
    positive_count = int(targets.sum().item())
    negative_count = len(targets) - positive_count
    if min(positive_count, negative_count) == 0:
        raise ValueError("CNN fitting requires both diagnostic classes")
    loss_function = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([negative_count / positive_count], dtype=torch.float32)
    )
    optimiser = torch.optim.Adam(
        model.parameters(), lr=learning_rate, weight_decay=cnn_weight_decay
    )
    loader = DataLoader(
        TensorDataset(images, targets),
        batch_size=cnn_batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
    )
    history = []
    validation_results = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch_images, batch_targets in loader:
            optimiser.zero_grad(set_to_none=True)
            logits = model(batch_images)
            loss = loss_function(logits, batch_targets)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite CNN loss at epoch {epoch}")
            loss.backward()
            optimiser.step()
            total_loss += loss.item() * len(batch_images)
        mean_loss = total_loss / len(images)
        history.append({"epoch": epoch, "training_loss": mean_loss})

        if epoch in cnn_epoch_options or epoch == epochs:
            message = f"{progress_label} | epoch {epoch}/{epochs} | loss {mean_loss:.4f}"
            if validation_data is not None:
                validation_images, metadata = validation_data
                model.eval()
                with torch.no_grad():
                    scores = torch.cat(
                        [model(batch) for batch in validation_images.split(cnn_batch_size)]
                    ).squeeze(1).numpy()
                if not np.isfinite(scores).all():
                    raise RuntimeError("CNN validation produced non-finite logits")
                image_results = metadata[["participant_id", "label"]].copy()
                image_results["decision_score"] = scores
                participant_results = (
                    image_results.groupby(["participant_id", "label"], as_index=False)
                    .agg(
                        images=("decision_score", "size"),
                        decision_score=("decision_score", "mean"),
                    )
                )
                if not participant_results["images"].eq(len(expected_tasks)).all():
                    raise RuntimeError("CNN validation requires nine drawings per participant")
                participant_score = balanced_accuracy_score(
                    participant_results["label"].map(label_to_target),
                    (participant_results["decision_score"] >= 0).astype(int),
                )
                validation_results.append(
                    {
                        "epochs": epoch,
                        "participant_balanced_accuracy": participant_score,
                        "image_balanced_accuracy": balanced_accuracy_score(
                            image_results["label"].map(label_to_target),
                            (scores >= 0).astype(int),
                        ),
                    }
                )
                message += f" | participant balanced accuracy {participant_score:.3f}"
            print(message, flush=True)

    model.eval()
    return model, history, validation_results


cnn_cv_rows = []
cnn_history_rows = []
cnn_tuning_start = perf_counter()
display(pd.DataFrame(cnn_fold_rows))
for learning_rate_index, learning_rate in enumerate(cnn_learning_rates):
    for fold_number, (fit_indices, validation_indices) in enumerate(cv_splits, start=1):
        fold_seed = random_state + fold_number
        cnn_fold_model, fold_history, fold_results = fit_cnn(
            cnn_training_images[fit_indices],
            cnn_training_targets[fit_indices],
            learning_rate=learning_rate,
            epochs=max(cnn_epoch_options),
            seed=fold_seed,
            validation_data=(
                cnn_training_images[validation_indices],
                training_metadata.iloc[validation_indices],
            ),
            progress_label=f"Learning rate {learning_rate:g}, fold {fold_number}/{len(cv_splits)}",
        )
        for result in fold_results:
            candidate_id = (
                learning_rate_index * len(cnn_epoch_options)
                + cnn_epoch_options.index(result["epochs"])
            )
            cnn_cv_rows.append(
                dict(result, candidate_id=candidate_id, learning_rate=learning_rate,
                     fold=fold_number, seed=fold_seed)
            )
        cnn_history_rows.extend(
            dict(row, learning_rate=learning_rate, fold=fold_number, seed=fold_seed)
            for row in fold_history
        )
cnn_tuning_seconds = perf_counter() - cnn_tuning_start

cnn_cv_fold_results = pd.DataFrame(cnn_cv_rows)
cnn_training_history = pd.DataFrame(cnn_history_rows)
cnn_cv_summary = (
    cnn_cv_fold_results.groupby(["candidate_id", "learning_rate", "epochs"], as_index=False)
    .agg(
        participant_balanced_accuracy_mean=("participant_balanced_accuracy", "mean"),
        participant_balanced_accuracy_std=("participant_balanced_accuracy", "std"),
        image_balanced_accuracy_mean=("image_balanced_accuracy", "mean"),
        folds=("fold", "nunique"),
    )
    .sort_values(
        ["participant_balanced_accuracy_mean", "participant_balanced_accuracy_std", "candidate_id"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)
if len(cnn_cv_summary) != len(cnn_learning_rates) * len(cnn_epoch_options):
    raise RuntimeError("The CNN search did not evaluate every candidate")
if not cnn_cv_summary["folds"].eq(len(cv_splits)).all():
    raise RuntimeError("A CNN candidate is missing a validation fold")

best_cnn_parameters = {
    "learning_rate": float(cnn_cv_summary.loc[0, "learning_rate"]),
    "epochs": int(cnn_cv_summary.loc[0, "epochs"]),
}
cnn_refit_start = perf_counter()
best_cnn, cnn_refit_history, _ = fit_cnn(
    cnn_training_images,
    cnn_training_targets,
    **best_cnn_parameters,
    seed=random_state,
    progress_label="Final training fit",
)
cnn_refit_seconds = perf_counter() - cnn_refit_start
cnn_refit_history = pd.DataFrame(cnn_refit_history)
if not all(torch.isfinite(parameter).all() for parameter in best_cnn.parameters()):
    raise RuntimeError("The fitted CNN contains non-finite weights")

print(f"Training images used: {len(cnn_training_images):,}")
print(f"Training participants used: {training_metadata['participant_id'].nunique():,}")
print(f"Selected CNN settings: {best_cnn_parameters}")
print(f"Tuning time: {cnn_tuning_seconds / 60:.2f} minutes")
print(f"Final fitting time: {cnn_refit_seconds / 60:.2f} minutes")
print("CNN test images used for fitting, tuning or scoring: 0")
display(cnn_cv_summary.round(4))

## Evaluate the CNN

Evaluate the selected CNN on the same held-out drawings and participants as the HOG–SVM model. Evaluation mode disables dropout, and gradient tracking is turned off. Drawing-level predictions use the fixed zero-logit threshold; participant-level predictions apply that threshold to the mean of each person's nine logits, matching the aggregation used during tuning. PD remains the positive class.

The metrics reuse the classical model's evaluation function. ROC AUC uses raw scores to assess ranking across thresholds; logits are not probabilities. The confusion matrix and participant results show the counts behind the percentages. This test set contains only 14 participants across 12 split groups, with some participants sharing exact duplicate images. These small counts limit the precision of performance estimates.

In [ ]:
cnn_test_metadata = (
    split_manifest.loc[split_manifest["partition"].eq("test")]
    .reset_index(drop=True)
)
if not np.array_equal(
    cnn_test_metadata["image_index"].to_numpy(), test_metadata["image_index"].to_numpy()
):
    raise RuntimeError("The CNN and HOG–SVM test images or their order differ")
for column in ["participant_id", "split_group", "sha256"]:
    if set(training_metadata[column]) & set(cnn_test_metadata[column]):
        raise RuntimeError(f"{column} overlap between CNN training and test data")
if cnn_test_metadata["label"].nunique() != 2:
    raise RuntimeError("CNN test evaluation requires both diagnostic classes")

cnn_test_images = torch.from_numpy(
    preprocessed_images[cnn_test_metadata["image_index"].to_numpy()]
).unsqueeze(1)
cnn_test_targets = cnn_test_metadata["label"].map(label_to_target).to_numpy()

best_cnn.eval()
with torch.no_grad():
    cnn_test_logits = torch.cat(
        [best_cnn(batch) for batch in cnn_test_images.split(cnn_batch_size)]
    )
if tuple(cnn_test_logits.shape) != (len(cnn_test_metadata), 1):
    raise RuntimeError("The CNN did not produce one test score per drawing")
if not torch.isfinite(cnn_test_logits).all():
    raise RuntimeError("CNN test scores contain non-finite values")
cnn_test_scores = cnn_test_logits.squeeze(1).cpu().numpy()
cnn_test_predictions = (cnn_test_scores >= 0).astype(int)

cnn_test_image_results = cnn_test_metadata[
    ["image_index", "participant_id", "label", "task", "split_group"]
].copy()
cnn_test_image_results["target"] = cnn_test_targets
cnn_test_image_results["decision_score"] = cnn_test_scores
cnn_test_image_results["prediction"] = cnn_test_predictions

cnn_test_participant_results = (
    cnn_test_image_results.groupby(["participant_id", "label"], as_index=False)
    .agg(
        images=("task", "size"),
        decision_score=("decision_score", "mean"),
        split_group=("split_group", "first"),
    )
)
if not cnn_test_participant_results["images"].eq(len(expected_tasks)).all():
    raise RuntimeError("A CNN test participant does not have nine evaluated drawings")
if len(cnn_test_participant_results) != cnn_test_metadata["participant_id"].nunique():
    raise RuntimeError("CNN aggregation changed the number of test participants")
cnn_test_participant_results["target"] = (
    cnn_test_participant_results["label"].map(label_to_target).astype(int)
)
cnn_test_participant_results["prediction"] = (
    cnn_test_participant_results["decision_score"] >= 0
).astype(int)
cnn_test_participant_results["predicted_label"] = (
    cnn_test_participant_results["prediction"].map({0: "Healthy", 1: "PD"})
)
cnn_test_participant_results["correct"] = (
    cnn_test_participant_results["target"]
    == cnn_test_participant_results["prediction"]
)

cnn_evaluation_metrics = pd.DataFrame(
    {
        "drawing level": calculate_binary_metrics(
            cnn_test_targets, cnn_test_predictions, cnn_test_scores
        ),
        "participant level": calculate_binary_metrics(
            cnn_test_participant_results["target"],
            cnn_test_participant_results["prediction"],
            cnn_test_participant_results["decision_score"],
        ),
    }
).T
cnn_participant_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        cnn_test_participant_results["target"],
        cnn_test_participant_results["prediction"],
        labels=[0, 1],
    ),
    index=["Actual Healthy", "Actual PD"],
    columns=["Predicted Healthy", "Predicted PD"],
)

print(f"Selected CNN settings: {best_cnn_parameters}")
print(f"Held-out test images: {len(cnn_test_metadata):,}")
print(f"Held-out test participants: {len(cnn_test_participant_results):,}")
print(f"Held-out split groups: {cnn_test_metadata['split_group'].nunique():,}")
display(cnn_evaluation_metrics.round(3))
display(cnn_participant_confusion_matrix)
display(
    cnn_test_participant_results[
        ["participant_id", "label", "decision_score", "predicted_label", "correct"]
    ].sort_values("decision_score")
)